# 01 · Preparação dos dados (Sprints 1 a 3)

Fase 3 do CRISP-DM. Lê os dados brutos do Inep, audita chaves e cobertura (Sprint 1), define escopo e rótulo (Sprint 2) e monta a tabela analítica escola × etapa (Sprint 3).

**Este notebook precisa dos dados brutos**, que não estão no repositório por tamanho. Eles ficam na pasta `DATASYNC_abandono_escolar_2025` do Google Drive do grupo, com a estrutura `01_censo/`, `02_rendimento/`, `03_nao_resposta/`, `05_indicadores_2025/`, `06_inse_2023/`, `07_ideb_2025/`. Para reproduzir sem o Drive, baixe as bases no portal de dados abertos do Inep e monte a mesma estrutura.

A saída final, `outputs/sprint3/tabela_analitica.csv`, é o arquivo publicado em `dados/tabela_analitica.csv`, ponto de partida dos notebooks 02 a 04.

## Sprint 1 · Auditoria inicial: arquivos, chaves e alvo

In [ ]:
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive')
PROJETO = Path('/content/drive/MyDrive/DATASYNC_abandono_escolar_2025')
if not (PROJETO / '01_censo').is_dir():
    raise FileNotFoundError(f'Pasta não encontrada: {PROJETO}. Confira o nome e a localização no Drive.')
os.chdir(PROJETO)
print('Pasta do projeto:', Path.cwd())

In [ ]:
from pathlib import Path
import csv
import re
import pandas as pd
from openpyxl import load_workbook

ROOT_OVERRIDE = None  # Ex.: Path('/content/drive/MyDrive/DATASYNC_abandono_escolar_2025')
candidates = [
    ROOT_OVERRIDE,
    Path.cwd(),
    Path.cwd().parent,
    Path('/Users/brainiac/Downloads/DATASYNC_abandono_escolar_2025'),
    Path('/content/drive/MyDrive/DATASYNC_abandono_escolar_2025'),
]
ROOT = next((p for p in candidates if p is not None and (p / '01_censo').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('Ajuste ROOT_OVERRIDE para a pasta DATASYNC_abandono_escolar_2025.')
ROOT = ROOT.resolve()
print('Pasta do projeto:', ROOT)

In [ ]:
censo_dir = ROOT / '01_censo/microdados_censo_escolar_2025_v2/dados'
censo_files = sorted(censo_dir.glob('*.csv'))
indicator_names = ['AFD', 'ATU', 'DSU', 'HAD', 'ICG', 'IED', 'IRD', 'TDI']
xlsx_files = [
    ROOT / '02_rendimento/tx_rend_escolas_2025/tx_rend_escolas_2025.xlsx',
    ROOT / '03_nao_resposta/tnr_escolas_2025/tnr_escolas_2025.xlsx',
    *(ROOT / f'05_indicadores_2025/{name}_2025_ESCOLAS/{name}_ESCOLAS_2025.xlsx'
      for name in indicator_names),
    ROOT / '06_inse_2023/INSE_2023_escolas.xlsx',
    ROOT / '07_ideb_2025/divulgacao_anos_iniciais_escolas_2025/divulgacao_anos_iniciais_escolas_2025.xlsx',
    ROOT / '07_ideb_2025/divulgacao_anos_finais_escolas_2025/divulgacao_anos_finais_escolas_2025.xlsx',
    ROOT / '07_ideb_2025/divulgacao_ensino_medio_escolas_2025/divulgacao_ensino_medio_escolas_2025.xlsx',
]
assert len(censo_files) == 6, f'Esperados 6 CSV do Censo; encontrados {len(censo_files)}.'
missing = [p for p in [*censo_files, *xlsx_files] if not p.is_file()]
assert not missing, f'Arquivos ausentes: {missing}'
print(f'Arquivos presentes: {len(censo_files)} CSV do Censo e {len(xlsx_files)} planilhas analíticas.')

In [ ]:
def workbook_header(path):
    wb = load_workbook(path, read_only=True, data_only=True)
    ws = wb.worksheets[0]
    found = None
    for number, row in enumerate(ws.iter_rows(min_row=1, max_row=15, values_only=True), 1):
        names = [str(x).strip() if x is not None else '' for x in row]
        if 'CO_ENTIDADE' in names or 'ID_ESCOLA' in names:
            found = (ws.title, number, names)
            break
    wb.close()
    if found is None:
        raise ValueError(f'Cabeçalho com chave de escola não encontrado: {path.name}')
    return found

inventory = []
for path in censo_files:
    with path.open(encoding='cp1252', newline='') as file:
        names = next(csv.reader(file, delimiter=';'))
    inventory.append({'fonte': path.stem, 'formato': 'CSV', 'aba': '',
                      'linha_cabecalho': 1, 'chave': 'CO_ENTIDADE',
                      'n_colunas': len(names), 'tamanho_mb': round(path.stat().st_size / 1e6, 1)})
for path in xlsx_files:
    sheet, number, names = workbook_header(path)
    key = 'CO_ENTIDADE' if 'CO_ENTIDADE' in names else 'ID_ESCOLA'
    inventory.append({'fonte': path.stem, 'formato': 'XLSX', 'aba': sheet,
                      'linha_cabecalho': number, 'chave': key,
                      'n_colunas': len(names), 'tamanho_mb': round(path.stat().st_size / 1e6, 1)})
inventory_df = pd.DataFrame(inventory)
print(inventory_df.to_string(index=False))

In [ ]:
def read_selected_xlsx(path, wanted):
    wb = load_workbook(path, read_only=True, data_only=True)
    ws = wb.worksheets[0]
    header_row = None
    positions = None
    for number, row in enumerate(ws.iter_rows(min_row=1, max_row=15, values_only=True), 1):
        names = [str(x).strip() if x is not None else '' for x in row]
        if all(name in names for name in wanted):
            header_row = number
            positions = {name: names.index(name) for name in wanted}
            break
    if positions is None:
        wb.close()
        raise ValueError(f'Colunas não encontradas em {path.name}: {wanted}')
    records = []
    for row in ws.iter_rows(min_row=header_row + 1, values_only=True):
        records.append({name: row[index] for name, index in positions.items()})
    wb.close()
    return pd.DataFrame.from_records(records, columns=wanted)

def school_code(value):
    if value is None or pd.isna(value):
        return None
    text = str(value).strip()
    if re.fullmatch(r'\d+\.0+', text):
        text = text.split('.')[0]
    return text if re.fullmatch(r'\d{8}', text) else None

rend_path = xlsx_files[0]
target_columns = ['NU_ANO_CENSO', 'CO_ENTIDADE', 'SG_UF', 'NO_CATEGORIA',
                  'NO_DEPENDENCIA', '3_CAT_FUN_AI', '3_CAT_FUN_AF', '3_CAT_MED']
rendimento = read_selected_xlsx(rend_path, target_columns)
print('Linhas de rodapé/fora de 2025:', int((~rendimento['NU_ANO_CENSO'].eq(2025)).sum()))
rendimento = rendimento.loc[rendimento['NU_ANO_CENSO'].eq(2025)].copy()
rendimento['codigo_escola'] = rendimento['CO_ENTIDADE'].map(school_code)
print('Linhas do rendimento:', len(rendimento))
print('Anos encontrados:', rendimento['NU_ANO_CENSO'].value_counts(dropna=False).to_dict())
print('Chaves inválidas:', rendimento['codigo_escola'].isna().sum())
print('Escolas duplicadas:', rendimento['codigo_escola'].dropna().duplicated().sum())
assert rendimento['NU_ANO_CENSO'].dropna().eq(2025).all()
assert rendimento['codigo_escola'].notna().all()
assert not rendimento['codigo_escola'].duplicated().any()

In [ ]:
target_names = {
    '3_CAT_FUN_AI': 'anos_iniciais',
    '3_CAT_FUN_AF': 'anos_finais',
    '3_CAT_MED': 'ensino_medio',
}
target_all = rendimento.melt(
    id_vars=['codigo_escola', 'SG_UF', 'NO_CATEGORIA', 'NO_DEPENDENCIA'],
    value_vars=list(target_names), var_name='coluna_origem', value_name='valor_original'
)
target_all['etapa'] = target_all['coluna_origem'].map(target_names)
target_all['taxa_abandono'] = pd.to_numeric(target_all['valor_original'], errors='coerce')
invalid_rate = target_all['taxa_abandono'].notna() & ~target_all['taxa_abandono'].between(0, 100)
print('Valores de taxa fora de 0–100:', int(invalid_rate.sum()))
print('Linhas sem taxa publicada:', int(target_all['taxa_abandono'].isna().sum()))
target_valid = target_all.loc[target_all['taxa_abandono'].notna() & ~invalid_rate].copy()
print('Linhas escola × etapa com taxa válida:', len(target_valid))
assert not target_valid.duplicated(['codigo_escola', 'etapa']).any()
summary = target_all.groupby('etapa').agg(
    linhas=('codigo_escola', 'size'),
    taxa_disponivel=('taxa_abandono', 'count'),
    taxa_zero=('taxa_abandono', lambda s: int(s.eq(0).sum())),
    minimo=('taxa_abandono', 'min'),
    maximo=('taxa_abandono', 'max'),
)
print(summary.to_string())

In [ ]:
school_csv = censo_dir / 'Tabela_Escola_2025_V2.csv'
censo = pd.read_csv(
    school_csv, sep=';', encoding='cp1252', dtype=str,
    usecols=['NU_ANO_CENSO', 'CO_ENTIDADE', 'SG_UF', 'TP_DEPENDENCIA', 'TP_LOCALIZACAO'],
    low_memory=False,
)
censo['codigo_escola'] = censo['CO_ENTIDADE'].map(school_code)
print('Censo — escolas:', len(censo), '| chaves inválidas:', censo['codigo_escola'].isna().sum(),
      '| duplicadas:', censo['codigo_escola'].dropna().duplicated().sum())
assert censo['NU_ANO_CENSO'].dropna().eq('2025').all()
assert censo['codigo_escola'].notna().all()
assert not censo['codigo_escola'].duplicated().any()

inse = read_selected_xlsx(xlsx_files[10], ['NU_ANO_SAEB', 'ID_ESCOLA', 'MEDIA_INSE', 'INSE_CLASSIFICACAO'])
print('Linhas do INSE fora de 2023:', int((~inse['NU_ANO_SAEB'].eq(2023)).sum()))
inse = inse.loc[inse['NU_ANO_SAEB'].eq(2023)].copy()
inse['codigo_escola'] = inse['ID_ESCOLA'].map(school_code)
print('INSE — escolas:', len(inse), '| chaves inválidas:', inse['codigo_escola'].isna().sum(),
      '| duplicadas:', inse['codigo_escola'].dropna().duplicated().sum())
assert inse['NU_ANO_SAEB'].dropna().eq(2023).all()
assert inse['codigo_escola'].notna().all()
assert not inse['codigo_escola'].duplicated().any()

In [ ]:
joined = target_valid.merge(
    censo[['codigo_escola', 'TP_DEPENDENCIA', 'TP_LOCALIZACAO']],
    on='codigo_escola', how='left', validate='m:1', indicator='_match_censo'
).merge(
    inse[['codigo_escola', 'MEDIA_INSE', 'INSE_CLASSIFICACAO']],
    on='codigo_escola', how='left', validate='m:1', indicator='_match_inse'
)
funnel = joined.groupby('etapa').agg(
    escola_etapa=('codigo_escola', 'size'),
    com_censo=('_match_censo', lambda s: int(s.eq('both').sum())),
    com_inse=('_match_inse', lambda s: int(s.eq('both').sum())),
    sem_censo=('_match_censo', lambda s: int(s.eq('left_only').sum())),
    sem_inse=('_match_inse', lambda s: int(s.eq('left_only').sum())),
)
funnel['censo_pct'] = (100 * funnel['com_censo'] / funnel['escola_etapa']).round(1)
funnel['inse_pct'] = (100 * funnel['com_inse'] / funnel['escola_etapa']).round(1)
print(funnel.to_string())
print('Checagem de unicidade escola × etapa:', not joined.duplicated(['codigo_escola', 'etapa']).any())

In [ ]:
output_dir = ROOT / 'outputs/sprint1'
output_dir.mkdir(parents=True, exist_ok=True)
inventory_df.to_csv(output_dir / 'inventario_fontes.csv', index=False, encoding='utf-8-sig')
target_valid[['codigo_escola', 'etapa', 'taxa_abandono', 'SG_UF', 'NO_CATEGORIA', 'NO_DEPENDENCIA']].to_csv(
    output_dir / 'alvo_escola_etapa_2025.csv', index=False, encoding='utf-8-sig'
)
funnel.to_csv(output_dir / 'cobertura_chaves.csv', encoding='utf-8-sig')
print('Arquivos gerados em:', output_dir)

## Sprint 2 · Escopo e rótulo

Rede pública, matrícula ≥ 10 na etapa. O rótulo `alvo_alto_abandono` (taxa ≥ P80 da etapa) é mantido na tabela só como referência: nos anos iniciais o P80 é zero e esse rótulo marca 100% das escolas. Os notebooks 03 e 04 recriam o alvo com a regra `taxa > 0` quando o limite é zero, e essa coluna **nunca** entra como preditora.

In [ ]:
from pathlib import Path
from openpyxl import load_workbook
import pandas as pd
import numpy as np
import re


alvo = pd.read_csv(
    ROOT / 'outputs/sprint1/alvo_escola_etapa_2025.csv',
    dtype={'codigo_escola': str}
)

def read_selected_xlsx(path, wanted):
    wb = load_workbook(path, read_only=True, data_only=True)
    ws = wb.worksheets[0]
    header_row = None
    positions = None
    for number, row in enumerate(ws.iter_rows(min_row=1, max_row=15, values_only=True), 1):
        names = [str(x).strip() if x is not None else '' for x in row]
        if all(name in names for name in wanted):
            header_row = number
            positions = {name: names.index(name) for name in wanted}
            break
    if positions is None:
        wb.close()
        raise ValueError(f'Colunas nao encontradas em {path.name}: {wanted}')
    records = []
    for row in ws.iter_rows(min_row=header_row + 1, values_only=True):
        records.append({name: row[index] for name, index in positions.items()})
    wb.close()
    return pd.DataFrame.from_records(records, columns=wanted)

def school_code(value):
    if value is None or pd.isna(value):
        return None
    text = str(value).strip()
    if re.fullmatch(r'\d+\.0+', text):
        text = text.split('.')[0]
    return text if re.fullmatch(r'\d{8}', text) else None

tnr_path = ROOT / '03_nao_resposta/tnr_escolas_2025/tnr_escolas_2025.xlsx'
tnr_cols = ['NU_ANO_CENSO', 'CO_ENTIDADE', '4_CAT_FUN_AI', '4_CAT_FUN_AF', '4_CAT_MED']
tnr_raw = read_selected_xlsx(tnr_path, tnr_cols)
tnr_raw = tnr_raw.loc[tnr_raw['NU_ANO_CENSO'].eq(2025)].copy()
tnr_raw['codigo_escola'] = tnr_raw['CO_ENTIDADE'].map(school_code)

tnr_map = {
    '4_CAT_FUN_AI': 'anos_iniciais',
    '4_CAT_FUN_AF': 'anos_finais',
    '4_CAT_MED': 'ensino_medio'
}
tnr_long = tnr_raw.melt(
    id_vars=['codigo_escola'],
    value_vars=list(tnr_map.keys()),
    var_name='coluna_tnr',
    value_name='valor_tnr'
)
tnr_long['etapa'] = tnr_long['coluna_tnr'].map(tnr_map)
tnr_long['tnr'] = pd.to_numeric(tnr_long['valor_tnr'], errors='coerce')

mat_path = ROOT / '01_censo/microdados_censo_escolar_2025_v2/dados/Tabela_Matricula_2025_V2.csv'
mat_cols = ['CO_ENTIDADE', 'QT_MAT_FUND_AI', 'QT_MAT_FUND_AF', 'QT_MAT_MED']
mat_raw = pd.read_csv(mat_path, sep=';', encoding='cp1252', dtype=str, usecols=mat_cols, low_memory=False)
mat_raw['codigo_escola'] = mat_raw['CO_ENTIDADE'].map(school_code)

mat_map = {
    'QT_MAT_FUND_AI': 'anos_iniciais',
    'QT_MAT_FUND_AF': 'anos_finais',
    'QT_MAT_MED': 'ensino_medio'
}
mat_long = mat_raw.melt(
    id_vars=['codigo_escola'],
    value_vars=list(mat_map.keys()),
    var_name='coluna_mat',
    value_name='valor_mat'
)
mat_long['etapa'] = mat_long['coluna_mat'].map(mat_map)
mat_long['matriculas'] = pd.to_numeric(mat_long['valor_mat'], errors='coerce')

df_sprint2 = alvo.merge(
    tnr_long[['codigo_escola', 'etapa', 'tnr']],
    on=['codigo_escola', 'etapa'],
    how='left',
    validate='1:1'
).merge(
    mat_long[['codigo_escola', 'etapa', 'matriculas']],
    on=['codigo_escola', 'etapa'],
    how='left',
    validate='1:1'
)

resumo_etapa = df_sprint2.groupby('etapa').agg(
    escolas=('codigo_escola', 'size'),
    taxa_zero_pct=('taxa_abandono', lambda s: round(100 * s.eq(0).mean(), 1)),
    P50=('taxa_abandono', lambda s: round(s.quantile(0.50), 2)),
    P75=('taxa_abandono', lambda s: round(s.quantile(0.75), 2)),
    P80=('taxa_abandono', lambda s: round(s.quantile(0.80), 2)),
    P90=('taxa_abandono', lambda s: round(s.quantile(0.90), 2)),
    tnr_acima_10_pct=('tnr', lambda s: round(100 * (s > 10).mean(), 1)),
    matricula_mediana=('matriculas', 'median')
).reset_index()

print("--- RESUMO ESTATISTICO POR ETAPA ---")
print(resumo_etapa.to_string(index=False))

df_publica = df_sprint2[
    df_sprint2['NO_DEPENDENCIA'].isin(['Estadual', 'Municipal', 'Federal']) & 
    (df_sprint2['matriculas'] >= 10)
].copy()

p80_dict = df_publica.groupby('etapa')['taxa_abandono'].quantile(0.80).to_dict()
print("\nLimiares P80 por etapa (Rede Publica):", p80_dict)

# P80 = 0 (Anos Iniciais): alto abandono = taxa > 0; demais etapas: taxa >= P80
df_publica['alvo_alto_abandono'] = df_publica.apply(
    lambda row: int(row['taxa_abandono'] > 0) if p80_dict.get(row['etapa'], 0) == 0
    else int(row['taxa_abandono'] >= p80_dict[row['etapa']]),
    axis=1
)

output_dir = ROOT / 'outputs' / 'sprint2'
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / 'base_sprint2_tratada.csv'
df_publica.to_csv(output_path, index=False, encoding='utf-8')
print("\nBase tratada guardada com sucesso em:", output_path)

## Sprint 3 · Tabela analítica escola × etapa

Junta indicadores do Inep, INSE 2023 e Censo 2025; cria as variáveis derivadas; registra as 8 verificações de qualidade e a matriz de papéis. Os nulos ficam na tabela: a imputação é feita dentro do pipeline de modelagem.

In [ ]:
from pathlib import Path
import re

from openpyxl import load_workbook
import numpy as np
import pandas as pd

SAIDA = ROOT / 'outputs' / 'sprint3'
CENSO = ROOT / '01_censo/microdados_censo_escolar_2025_v2/dados'
CHAVE = ['codigo_escola', 'etapa']

# etapa: (sufixo nos indicadores, sufixo no Censo)
ETAPAS = {
    'anos_iniciais': ('FUN_AI', 'FUND_AI'),
    'anos_finais': ('FUN_AF', 'FUND_AF'),
    'ensino_medio': ('MED', 'MED'),
}

INFRA = ['IN_AGUA_POTAVEL', 'IN_ENERGIA_REDE_PUBLICA', 'IN_ESGOTO_REDE_PUBLICA', 'IN_BIBLIOTECA_SALA_LEITURA',
         'IN_LABORATORIO_CIENCIAS', 'IN_LABORATORIO_INFORMATICA', 'IN_QUADRA_ESPORTES', 'IN_INTERNET_ALUNOS']

achados = []
linhas_sem_codigo = 0


def school_code(value):
    if value is None or pd.isna(value):
        return None
    text = str(value).strip()
    if re.fullmatch(r'\d+\.0+', text):
        text = text.split('.')[0]
    return text if re.fullmatch(r'\d{8}', text) else None


def ler_xlsx(path, colunas):
    wb = load_workbook(path, read_only=True, data_only=True)
    ws = wb.worksheets[0]
    posicoes = None
    for numero, row in enumerate(ws.iter_rows(min_row=1, max_row=15, values_only=True), 1):
        nomes = [str(x).strip() if x is not None else '' for x in row]
        if all(c in nomes for c in colunas):
            posicoes = [nomes.index(c) for c in colunas]
            break
    if posicoes is None:
        wb.close()
        raise ValueError(f'Colunas nao encontradas em {path.name}: {colunas}')
    dados = [[row[i] for i in posicoes] for row in ws.iter_rows(min_row=numero + 1, values_only=True)]
    wb.close()
    return pd.DataFrame(dados, columns=colunas)


def preparar_chave(df, coluna='CO_ENTIDADE'):
    global linhas_sem_codigo
    df = df.copy()
    df['codigo_escola'] = df[coluna].map(school_code)
    linhas_sem_codigo += int(df['codigo_escola'].isna().sum())
    return df[df['codigo_escola'].notna()].drop(columns=coluna)


def indicador(sigla):
    return ROOT / f'05_indicadores_2025/{sigla}_2025_ESCOLAS/{sigla}_ESCOLAS_2025.xlsx'


def por_etapa(df, modelos, posicao_sufixo):
    """Transforma colunas por etapa em uma linha por escola x etapa."""
    partes = []
    for etapa, sufixos in ETAPAS.items():
        nomes = {modelo.format(sufixos[posicao_sufixo]): novo for novo, modelo in modelos.items()}
        partes.append(df[['codigo_escola', *nomes]].rename(columns=nomes).assign(etapa=etapa))
    return pd.concat(partes, ignore_index=True)


def ler_censo(arquivo, colunas):
    df = pd.read_csv(CENSO / arquivo, sep=';', encoding='cp1252', dtype=str, usecols=['CO_ENTIDADE', *colunas])
    return preparar_chave(df)


def razao(numerador, denominador):
    return (numerador / denominador).where(denominador > 0)


def achado(numero, verificacao, resultado, acao):
    achados.append({'numero': numero, 'verificacao': verificacao, 'resultado': resultado, 'acao': acao})


# --- Leitura e juncao ---------------------------------------------------------

base = pd.read_csv(ROOT / 'outputs/sprint2/base_sprint2_tratada.csv', dtype={'codigo_escola': str})
chaves_repetidas = int(base.duplicated(CHAVE).sum())
tabela = base.copy()

INDICADORES_ETAPA = {
    'ATU': {'atu_alunos_turma': '{}_CAT_0'},
    'HAD': {'had_horas_aula': '{}_CAT_0'},
    'DSU': {'dsu_doc_superior': '{}_CAT_0'},
    'AFD': {'afd_formacao_adequada': '{}_CAT_1'},
    'IED': {'ied_nivel5': '{}_CAT_5', 'ied_nivel6': '{}_CAT_6'},
    'TDI': {'tdi_distorcao': '{}_CAT_0'},
}
for sigla, modelos in INDICADORES_ETAPA.items():
    colunas = [m.format(s[0]) for m in modelos.values() for s in ETAPAS.values()]
    df = preparar_chave(ler_xlsx(indicador(sigla), ['CO_ENTIDADE', *colunas]))
    tabela = tabela.merge(por_etapa(df, modelos, 0), on=CHAVE, how='left', validate='1:1')
    print('Lido:', sigla, flush=True)

por_escola = [
    (ler_xlsx(indicador('ICG'), ['CO_ENTIDADE', 'COMPLEX']), 'CO_ENTIDADE', {'COMPLEX': 'icg_complexidade'}),
    (ler_xlsx(indicador('IRD'), ['CO_ENTIDADE', 'EDU_BAS_CAT_0']), 'CO_ENTIDADE', {'EDU_BAS_CAT_0': 'ird_regularidade'}),
    (ler_xlsx(ROOT / '06_inse_2023/INSE_2023_escolas.xlsx', ['ID_ESCOLA', 'MEDIA_INSE']), 'ID_ESCOLA', {'MEDIA_INSE': 'inse_media'}),
]
for df, coluna_chave, nomes in por_escola:
    df = preparar_chave(df, coluna_chave).rename(columns=nomes)
    tabela = tabela.merge(df, on='codigo_escola', how='left', validate='m:1')
print('Lidos: ICG, IRD, INSE', flush=True)

escola = ler_censo('Tabela_Escola_2025_V2.csv', ['CO_MUNICIPIO', *INFRA])
tabela = tabela.merge(escola, on='codigo_escola', how='left', validate='m:1')

modelos_mat = {'mat_noturno': 'QT_MAT_{}_N', 'mat_integral': 'QT_MAT_{}_INT'}
colunas_mat = [m.format(s[1]) for m in modelos_mat.values() for s in ETAPAS.values()]
matricula = ler_censo('Tabela_Matricula_2025_V2.csv', ['QT_MAT_BAS', 'QT_TRANSP_PUBLICO', *colunas_mat])
tabela = tabela.merge(matricula[['codigo_escola', 'QT_MAT_BAS', 'QT_TRANSP_PUBLICO']], on='codigo_escola', how='left', validate='m:1')
tabela = tabela.merge(por_etapa(matricula, modelos_mat, 1), on=CHAVE, how='left', validate='1:1')

for arquivo, nome, prefixo in [('Tabela_Docente_2025_V2.csv', 'doc_etapa', 'QT_DOC_'),
                               ('Tabela_Turma_2025_V2.csv', 'tur_etapa', 'QT_TUR_')]:
    colunas = [prefixo + s[1] for s in ETAPAS.values()]
    df = ler_censo(arquivo, colunas)
    tabela = tabela.merge(por_etapa(df, {nome: prefixo + '{}'}, 1), on=CHAVE, how='left', validate='1:1')
print('Lido: Censo', flush=True)

# --- Verificacoes e tratamento ------------------------------------------------

achado(1, 'Chave com 8 digitos', f'{linhas_sem_codigo} linhas sem codigo valido nas fontes', 'Linhas descartadas (rodapes)')
achado(2, 'Chave (escola, etapa) unica na base', f'{chaves_repetidas} repeticoes', 'Juncoes com validate; teste final abaixo')

tabela['icg_complexidade'] = tabela['icg_complexidade'].astype('string').str.extract(r'(\d)', expand=False)
numericas = [c for c in tabela.columns if c not in
             ['codigo_escola', 'etapa', 'SG_UF', 'NO_CATEGORIA', 'NO_DEPENDENCIA', 'CO_MUNICIPIO']]
marcadores = int(tabela[numericas].astype('string').apply(lambda s: s.str.strip().eq('--')).sum().sum())
achado(3, "Marcador de ausencia '--'", f'{marcadores} celulas', 'Convertidas em nulo, sem imputar')

antes = tabela[numericas].notna()
texto = tabela[numericas].astype('string').apply(lambda s: s.str.strip().eq('--'))
tabela[numericas] = tabela[numericas].apply(pd.to_numeric, errors='coerce')
nao_numericos = int((antes & ~texto & tabela[numericas].isna()).sum().sum())
achado(4, 'Texto em coluna numerica', f'{nao_numericos} celulas', 'Convertidas em nulo')

percentuais = ['taxa_abandono', 'tnr', 'dsu_doc_superior', 'afd_formacao_adequada', 'tdi_distorcao', 'ied_nivel5', 'ied_nivel6']
fora = tabela[percentuais].lt(0) | tabela[percentuais].gt(100)
tabela[percentuais] = tabela[percentuais].mask(fora)
achado(5, 'Percentuais fora de 0-100', f'{int(fora.sum().sum())} celulas', 'Convertidas em nulo')

sem_turma = tabela['tur_etapa'].eq(0)
achado(6, 'Matricula na etapa sem turma registrada', f'{int(sem_turma.sum())} linhas', 'Mantidas; criada flag_sem_turma_etapa')

# --- Variaveis derivadas ------------------------------------------------------

tabela['log_matriculas'] = np.log1p(tabela['matriculas'])
tabela['pct_noturno'] = 100 * razao(tabela['mat_noturno'], tabela['matriculas'])
tabela['pct_integral'] = 100 * razao(tabela['mat_integral'], tabela['matriculas'])
tabela['pct_transporte_publico'] = 100 * razao(tabela['QT_TRANSP_PUBLICO'], tabela['QT_MAT_BAS'])
tabela['alunos_por_docente'] = razao(tabela['matriculas'], tabela['doc_etapa'])
tabela['indice_infra'] = tabela[INFRA].mean(axis=1)
tabela['ied_esforco_alto'] = tabela[['ied_nivel5', 'ied_nivel6']].sum(axis=1, min_count=2)
tabela['flag_sem_turma_etapa'] = sem_turma.astype(int)
tabela['flag_inse_ausente'] = tabela['inse_media'].isna().astype(int)

tabela = tabela.drop(columns=[*INFRA, 'ied_nivel5', 'ied_nivel6', 'mat_noturno', 'mat_integral',
                              'QT_MAT_BAS', 'QT_TRANSP_PUBLICO', 'doc_etapa', 'tur_etapa'])

cobertura = tabela.drop(columns=CHAVE).notna().groupby(tabela['etapa']).mean().mul(100).round(1)
abaixo = cobertura.stack()
abaixo = abaixo[abaixo < 90]
achado(7, 'Cobertura abaixo de 90% por etapa', '; '.join(f'{v} em {e}: {p}%' for (e, v), p in abaixo.items()) or 'nenhuma',
       'Mantidas com nulo; imputar na Sprint 5')

prevalencia = tabela.groupby('etapa')['alvo_alto_abandono'].mean().mul(100).round(1)
achado(8, 'Prevalencia do alvo por etapa', '; '.join(f'{e}: {p}%' for e, p in prevalencia.items()),
       'Anos iniciais em 100%: rotulo a rever pelo grupo')

# --- Matriz de papeis ---------------------------------------------------------

PAPEIS = [
    ('codigo_escola', 'chave', 'Identifica a escola'),
    ('etapa', 'chave', 'Identifica a etapa'),
    ('alvo_alto_abandono', 'alvo', 'Taxa >= P80 da etapa (Sprint 2)'),
    ('taxa_abandono', 'alvo', 'Origem do rotulo; so para testar outros limiares'),
    ('CO_MUNICIPIO', 'grupo de validacao', 'Agrupa as dobras na Sprint 5'),
    ('matriculas', 'condicional', 'Usado no filtro; no modelo entra o log'),
    ('tnr', 'condicional', 'Qualidade do dado; analise de sensibilidade'),
    ('tdi_distorcao', 'condicional', 'Acumula atraso escolar; testar com e sem na Sprint 5'),
    ('SG_UF', 'preditora', 'Contexto regional'),
    ('NO_CATEGORIA', 'preditora', 'Urbana ou rural'),
    ('NO_DEPENDENCIA', 'preditora', 'Rede'),
    ('atu_alunos_turma', 'preditora', 'Alunos por turma'),
    ('had_horas_aula', 'preditora', 'Horas-aula diarias'),
    ('dsu_doc_superior', 'preditora', 'Docentes com superior'),
    ('afd_formacao_adequada', 'preditora', 'Docentes com formacao adequada (grupo 1)'),
    ('ied_esforco_alto', 'preditora', 'Docentes nos niveis 5 e 6 de esforco'),
    ('icg_complexidade', 'preditora', 'Complexidade de gestao'),
    ('ird_regularidade', 'preditora', 'Regularidade docente'),
    ('inse_media', 'preditora', 'Nivel socioeconomico de 2023'),
    ('flag_inse_ausente', 'preditora', 'INSE ausente'),
    ('log_matriculas', 'preditora', 'Porte da etapa'),
    ('pct_noturno', 'preditora', 'Matriculas noturnas'),
    ('pct_integral', 'preditora', 'Matriculas em tempo integral'),
    ('pct_transporte_publico', 'preditora', 'Uso de transporte publico'),
    ('alunos_por_docente', 'preditora', 'Alunos por docente da etapa'),
    ('indice_infra', 'preditora', 'Proporcao de 8 itens de infraestrutura'),
    ('flag_sem_turma_etapa', 'preditora', 'Etapa sem turma registrada'),
]
EXCLUIDAS = [
    ('aprovacao e reprovacao 2025', 'excluida', 'Somadas ao abandono dao 100%: vazamento'),
    ('Ideb 2025 e rendimento P', 'excluida', 'Calculados com a aprovacao de 2025: vazamento'),
    ('notas do Saeb 2025', 'excluida', 'Componente do Ideb; existe so para parte das escolas'),
]

sem_papel = set(tabela.columns) - {p[0] for p in PAPEIS}
assert not sem_papel, f'Colunas sem papel: {sem_papel}'
proibidas = [p[0] for p in PAPEIS if p[1] == 'preditora' and re.search('aprov|reprov|aband|ideb|rend|saeb', p[0], re.I)]
assert not proibidas, f'Possivel vazamento: {proibidas}'
assert not tabela.duplicated(CHAVE).any() and len(tabela) == len(base), 'Chave nao unica ou linhas alteradas'


def maior_spearman(variavel):
    if variavel in ('taxa_abandono', 'alvo_alto_abandono') or not pd.api.types.is_numeric_dtype(tabela[variavel]):
        return None
    valores = []
    for _, grupo in tabela.groupby('etapa'):
        par = grupo[[variavel, 'taxa_abandono']].dropna()
        if par[variavel].nunique() > 1:
            valores.append(par[variavel].rank().corr(par['taxa_abandono'].rank()))
    return round(max(valores, key=abs), 3) if valores else None


matriz = pd.DataFrame(PAPEIS + EXCLUIDAS, columns=['variavel', 'papel', 'motivo'])
matriz['maior_spearman_com_taxa'] = [maior_spearman(v) if v in tabela else None for v in matriz['variavel']]

SAIDA.mkdir(parents=True, exist_ok=True)
tabela.to_csv(SAIDA / 'tabela_analitica.csv', index=False, encoding='utf-8-sig')
pd.DataFrame(achados).to_csv(SAIDA / 'achados_qualidade.csv', index=False, encoding='utf-8-sig')
matriz.to_csv(SAIDA / 'matriz_papeis.csv', index=False, encoding='utf-8-sig')

print(pd.DataFrame(achados).to_string(index=False))
print(tabela.groupby('etapa').size().to_string())
print(matriz.dropna(subset=['maior_spearman_com_taxa']).sort_values('maior_spearman_com_taxa', key=abs, ascending=False).head(5).to_string(index=False))
print('Preditoras:', (matriz['papel'] == 'preditora').sum(), '| Tabela:', tabela.shape)

In [ ]:
# Conferência com o arquivo publicado no repositório
final = pd.read_csv(SAIDA / "tabela_analitica.csv", dtype={"codigo_escola": str})
print("Linhas e colunas:", final.shape, "(esperado: (136713, 27))")
assert final.shape == (136713, 27)